# Run the 2D + 3D pipelines on Colab

Thin Colab orchestrator around `scripts/run_2D_pipeline.py` and `scripts/run_3D_reconstruction_pipeline.py`. Designed to run on a Colab GPU runtime so the YOLO inference can use larger batches than a laptop allows.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is sufficient).
2. Place your synchronised match videos at `/content/drive/MyDrive/cv-project/videos/out2.mp4`, `out4.mp4`, `out13.mp4` — or upload them directly in Step 5.
3. (Optional) Place calibration JSONs at `/content/drive/MyDrive/cv-project/camera_data/cam_*.json` if you have already calibrated extrinsics.

Fine-tuned weights are auto-downloaded from Hugging Face on first run.

## 1. Verify GPU runtime

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless huggingface_hub motmetrics trackeval matplotlib python-dotenv

## 3. Clone the repo

In [ ]:
!git clone https://github.com/446f6e6e79/player-tracking-in-sports.git
%cd player-tracking-in-sports

## 4. Mount Google Drive (optional)

Skip this cell and use the upload cell instead if you do not want to use Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Wire input directories

Set the paths the pipelines should read from. Defaults below assume the Drive layout from the intro; uncomment the upload block if you would rather upload from your machine for this single session.

In [ ]:
from pathlib import Path
import shutil

VIDEOS_INPUT_DIR = '/content/drive/MyDrive/cv-project/videos'
CAMERA_DATA_DIR = '/content/drive/MyDrive/cv-project/camera_data'
OUTPUT_DIR = '/content/results'

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Copy camera calibration JSONs into the repo's expected location so the
# defaults in src/paths/defaults.py keep working.
if Path(CAMERA_DATA_DIR).exists():
    target = Path('data/camera_data')
    target.mkdir(parents=True, exist_ok=True)
    for src in Path(CAMERA_DATA_DIR).glob('cam_*.json'):
        shutil.copy(src, target / src.name)
    print(f'Copied calibration files into {target}')

# Confirm the videos are reachable.
for name in ('out2.mp4', 'out4.mp4', 'out13.mp4'):
    p = Path(VIDEOS_INPUT_DIR) / name
    print(p, '-', 'OK' if p.exists() else 'MISSING')

# Optional alternative: uncomment to upload the three videos from your machine.
# from google.colab import files
# uploaded = files.upload()
# VIDEOS_INPUT_DIR = '/content'
# for fname in uploaded:
#     print('Uploaded', fname)

## 6. Run the 2D pipeline on each camera

Calls `run_2d_pipeline` programmatically (no subprocess) so you can tweak the GPU-friendly defaults inline. On a T4 you can comfortably push `yolo_batch_size` to 16 or 32; `chunk_size` controls peak frame RAM.

In [ ]:
import sys
sys.path.insert(0, '/content/player-tracking-in-sports')

from scripts.run_2D_pipeline import run_2d_pipeline

CAMERAS = ('cam_13', 'cam_4', 'cam_2')
YOLO_BATCH = 16    # T4-friendly. Increase to 32 on an A100; drop to 4 on laptops.
CHUNK_SIZE = 256   # 256 BGR 1080p frames ≈ 1.5 GB; well under T4 host RAM.

for cam in CAMERAS:
    print(f'\n=== {cam} ===')
    run_2d_pipeline(
        camera=cam,
        input_dir=VIDEOS_INPUT_DIR,
        output_dir=OUTPUT_DIR,
        yolo_batch_size=YOLO_BATCH,
        chunk_size=CHUNK_SIZE,
        force=True,
    )

## 7. Run the 3D reconstruction pipeline

Triangulates the per-camera tracking JSONs from Step 6 and writes `triangulation.json`. Toggle the render flags to also produce the minimap MP4, radar overlay on camera A, or 3D matplotlib animation.

In [ ]:
from scripts.run_3D_reconstruction_pipeline import run_3d_reconstruction_pipeline

run_3d_reconstruction_pipeline(
    cameras=CAMERAS,
    output_dir=OUTPUT_DIR,
    render_minimap=True,
    overlay_video=False,    # set True to render the radar overlay on cam A
    render_3d_graph=False,  # set True for the matplotlib 3D animation
    force=True,
)

## 8. Persist the outputs

Copy the produced JSONs and videos to Drive (so you do not lose them when the runtime dies). Replace the destination if you prefer a different location.

In [ ]:
import shutil

DRIVE_DEST = '/content/drive/MyDrive/cv-project/results'
Path(DRIVE_DEST).mkdir(parents=True, exist_ok=True)

if Path('/content/drive').exists():
    shutil.copytree(OUTPUT_DIR, DRIVE_DEST, dirs_exist_ok=True)
    print(f'Copied {OUTPUT_DIR} to {DRIVE_DEST}')
else:
    # Fallback: trigger a browser download of the resolved tracking video for cam_13.
    from google.colab import files
    files.download(f'{OUTPUT_DIR}/cam_13/videos/tracking_resolved.mp4')